In [1]:
# Standard library imports
import os
import sys

# Third-party imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

In [2]:
postprocessed_data_path = BASE_OUTPUT_PATH / 'postprocessed_data'

ex_statistical = pd.read_csv(postprocessed_data_path / 'expanding_window_statistical_models_comparison.csv')
rw_statistical = pd.read_csv(postprocessed_data_path / 'rolling_window_statistical_models_comparison.csv')

ex_lstm = pd.read_csv(postprocessed_data_path / 'expanding_window_lstm_models_comparison.csv')
ex_lstm_42 = ex_lstm[ex_lstm['MODEL'].str.contains('ts42')].reset_index(drop=True)
ex_lstm_168 = ex_lstm[ex_lstm['MODEL'].str.contains('ts168')].reset_index(drop=True)

In [3]:
new_names = {
    'rw_arma_3m': 'ARMA (3 Months)',
    'rw_arma_6m': 'ARMA (6 Months)',
    'rw_arma_12m': 'ARMA (12 Months)',
    'rw_armax_3m': 'ARMAX (3 Months)',
    'rw_armax_6m': 'ARMAX (6 Months)',
    'rw_armax_12m': 'ARMAX (12 Months)',
    'rw_sarma_3m': 'SARMA (3 Months)',
    'rw_sarma_6m': 'SARMA (6 Months)',
    'rw_sarma_12m': 'SARMA (12 Months)',
    'rw_sarmax_3m': 'SARMAX (3 Months)',
    'rw_sarmax_6m': 'SARMAX (6 Months)',
    'rw_sarmax_12m': 'SARMAX (12 Months)',
    'arma': 'ARMA',
    'armax': 'ARMAX',
    'sarma': 'SARMA',
    'sarmax': 'SARMAX',
    'lstm_ts42_single': 'LSTM (Single)',
    'lstm_ts42_ensemble': 'LSTM (Ensemble)',
    'lstm_ts42_recency': 'LSTM (Weighted Ensemble)',
    'lstm_ts168_single': 'LSTM (Single)',
    'lstm_ts168_ensemble': 'LSTM (Ensemble)',
    'lstm_ts168_recency': 'LSTM (Weighted Ensemble)',
    'lstm_feature_ts42_single': 'LSTM with Feature (Single)',
    'lstm_feature_ts42_ensemble': 'LSTM with Feature (Ensemble)',
    'lstm_feature_ts42_recency': 'LSTM with Feature (Weighted Ensemble)',
    'lstm_feature_ts168_single': 'LSTM with Feature (Single)',
    'lstm_feature_ts168_ensemble': 'LSTM with Feature (Ensemble)',
    'lstm_feature_ts168_recency': 'LSTM with Feature (Weighted Ensemble)',
    'bidirectional_lstm_ts42_single': 'BiLSTM (Single)',
    'bidirectional_lstm_ts42_ensemble': 'BiLSTM (Ensemble)',
    'bidirectional_lstm_ts42_recency': 'BiLSTM (Weighted Ensemble)',
    'bidirectional_lstm_ts168_single': 'BiLSTM (Single)',
    'bidirectional_lstm_ts168_ensemble': 'BiLSTM (Ensemble)',
    'bidirectional_lstm_ts168_recency': 'BiLSTM (Weighted Ensemble)',
}

In [4]:
def find_info_of_highest(dataset, metric):
    """
    Finds and returns the information about the row with the highest value for a specified metric in the dataset.
    
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset to search through.
    metric : str
        The column name of the metric to find the maximum value for.
        
    Returns:
    --------
    dict
        A dictionary containing relevant information about the row with the highest metric value.
        Keys include: 'MODEL', 'MODEL_NAME', 'PRODUCT', 'PRODUCT_SIGN', and 'PRICE_TYPES'.
    """
    # Find the index of the row with the maximum value for the specified metric
    max_idx = dataset[metric].idxmax()
    
    # Extract the row with the highest metric value
    highest_row = dataset.loc[max_idx]

    # Create a dictionary with the relevant information from the highest-performing row
    highest = {
        'MODEL': highest_row['MODEL'],
        'MODEL_NAME': highest_row['MODEL_NAME'],
        'PRODUCT': highest_row['PRODUCT'],
        'PRODUCT_SIGN': highest_row['PRODUCT_SIGN'],
        'PRICE_TYPES': highest_row['PRICE_TYPE']
    }
    return highest

In [ ]:
best_ex_statistical = find_info_of_highest(ex_statistical, 'REVENUE')
best_rw_statistical = find_info_of_highest(rw_statistical, 'REVENUE')

best_ex_lstm_42 = find_info_of_highest(ex_lstm_42, 'REVENUE')
best_ex_lstm_168 = find_info_of_highest(ex_lstm_168, 'REVENUE')

print(f"Best Expanding Window Statistical Model: {best_ex_statistical}")
print(f"Best Rolling Window Statistical Model: {best_rw_statistical}")
print(f"Best Expanding Window LSTM Model (42): {best_ex_lstm_42}")
print(f"Best Expanding Window LSTM Model (168): {best_ex_lstm_168}")

In [6]:
def find_and_read_file(model, product, product_sign, price_type, output_path):
    """
    Searches for a CSV file with a specific naming pattern and reads it into a pandas DataFrame.
    The function recursively searches through the specified output path to find a file that 
    exactly matches the generated pattern, and returns the content as a DataFrame with 'DATE' 
    column parsed as datetime and set as index.
    Parameters:
    -----------
    model : str
        Model name to include in the file pattern.
    product : str
        Product name to include in the file pattern.
    product_sign : str
        Product sign identifier to include in the file pattern.
    price_type : str
        Price type to include in the file pattern.
    output_path : str
        Directory path to search for the file.
    Returns:
    --------
    pandas.DataFrame or None
        DataFrame containing the file contents with DATE as index if found,
        None if the file doesn't exist or the output_path is invalid.
    Notes:
    ------
    The file naming pattern is: 
    '{model}_model_processed_results_{product}_{product_sign}_{price_type}.csv'
    """
    name_pattern = f'{model}_model_processed_results_{product}_{product_sign}_{price_type}.csv'

    # Walking through output_path to find the file
    if os.path.exists(output_path):
        for root, dirs, files in os.walk(output_path):
            for file in files:
                if file == name_pattern:  # Use exact match instead of substring match
                    full_path = os.path.join(root, file)
                    print(f"Found file at: {full_path}")
                    return pd.read_csv(full_path, parse_dates=['DATE'], index_col=['DATE'])
        
        # If we reach here, we haven't found the file anywhere in output_path
        print(f"File not found in any subdirectory: {name_pattern}")
        return None
    else:
        print(f"Output path does not exist: {output_path}")
        return None

In [ ]:
# Group models into categories for easier processing
model_groups = {
    'ROLLING_WINDOW_STATISTICAL':   ['rw_arma_3m', 'rw_arma_6m', 'rw_arma_12m',
                                     'rw_armax_3m', 'rw_armax_6m', 'rw_armax_12m',
                                     'rw_sarma_3m', 'rw_sarma_6m', 'rw_sarma_12m',
                                     'rw_sarmax_3m', 'rw_sarmax_6m', 'rw_sarmax_12m'],
    'EXPANDING_WINDOW_STATISTICAL': ['arma', 'armax', 'sarma', 'sarmax'],
    'EXPANDING_WINDOW_LSTM_42':     ['lstm_ts42_single', 'lstm_ts42_ensemble', 'lstm_ts42_recency',
                                     'lstm_feature_ts42_single', 'lstm_feature_ts42_ensemble', 'lstm_feature_ts42_recency',
                                     'bidirectional_lstm_ts42_single', 'bidirectional_lstm_ts42_ensemble', 'bidirectional_lstm_ts42_recency'],
    'EXPANDING_WINDOW_LSTM_168':    ['lstm_ts168_single', 'lstm_ts168_ensemble', 'lstm_ts168_recency',
                                     'lstm_feature_ts168_single', 'lstm_feature_ts168_ensemble', 'lstm_feature_ts168_recency',
                                     'bidirectional_lstm_ts168_single', 'bidirectional_lstm_ts168_ensemble', 'bidirectional_lstm_ts168_recency']
}
# Define price types to process
price_types = ['AVERAGE', 'MARGINAL']

# Iterate through each price type and model group to collect and organize results
for price_type in price_types:
    for model_group in model_groups:
        model_group_df = None
        
        # Process each model within the current model group
        for model in model_groups[model_group]:
            # Find and read the model results file
            model_df = find_and_read_file(
                model, 
                best_ex_statistical['PRODUCT'], 
                best_ex_statistical['PRODUCT_SIGN'], 
                price_type, 
                BASE_OUTPUT_PATH
            )
            
            if model_df is not None:
                # Convert DATE to datetime if it's a string column (safety check)
                if 'DATE' in model_df.columns:
                    model_df['DATE'] = pd.to_datetime(model_df['DATE'])
                    model_df.set_index('DATE', inplace=True)
                
                # Add model identifiers as columns
                model_df['MODEL'] = model
                model_df['MODEL_NAME'] = new_names.get(model, model)
                
                # Rename the prediction and error metric columns to include the model name for comparison
                model_df = model_df.rename(columns={'D+1': f'D+1_{model}',
                                                    'RMSE': f'RMSE_{model}',
                                                    'MAPE': f'MAPE_{model}',
                                                    'MAE': f'MAE_{model}'})
                
                if model_group_df is None:
                    # First model in the group, use as base dataframe
                    model_group_df = model_df[['ACTUAL_VALUE', 'MARGINAL_PRICE', 'MODEL', 'MODEL_NAME', 
                                            f'D+1_{model}', f'RMSE_{model}', f'MAPE_{model}', f'MAE_{model}']].copy()
                else:
                    # Join with the existing dataframe on index (DATE) to add this model's columns
                    model_group_df = model_group_df.join(model_df[[f'D+1_{model}', f'RMSE_{model}', f'MAPE_{model}', f'MAE_{model}']])
        
        # Save the combined results for this model group if data was found
        if model_group_df is not None:
            model_group_df.to_csv(
                postprocessed_data_path / f'{model_group.lower()}_{best_ex_statistical["PRODUCT"]}_{best_ex_statistical["PRODUCT_SIGN"]}_{price_type}_results.csv',
                index=True
            )

In [8]:
def plot_histogram(dataset, model, product, product_sign, price_type, output_path, rw=None):
    """
    Creates and saves a histogram visualization of MAPE (Mean Absolute Percentage Error) values
    for energy price forecasting models.
    The histogram shows the distribution of MAPE values for the specified model, with vertical lines
    indicating median MAPE values for the current model and other models in the dataset for comparison.
    Parameters
    ----------
    dataset : pandas.DataFrame
        DataFrame containing MAPE values for different models.
    model : str
        The main model to visualize (e.g., 'LSTM', 'GRU', 'Naive').
    product : str
        Energy product type being modeled (e.g., 'electricity', 'gas').
    product_sign : str
        Direction indicator (e.g., 'buy', 'sell').
    price_type : str
        Type of price being analyzed (e.g., 'spot', 'forward').
    output_path : pathlib.Path
        Directory path where the output histogram will be saved.
    rw : int or None, optional
        Rolling window size filter. If provided, only includes MAPE columns
        with this rolling window value.
    Returns
    -------
    None
        The function saves the histogram plot to the specified output path and closes the figure.
    Notes
    -----
    - The histogram bins range from 0 to 100% in increments of 10%.
    - The main model's median is shown with a solid blue line.
    - Comparison models are shown with dashed lines in different colors.
    - The legend is sorted by median MAPE values (ascending).
    - Special handling for models with timestep indicators (ts42, ts168).
    - Uses the 'new_names' dictionary for model name display formatting.
    """
    plt.figure(figsize=(12, 12))

    # Determine if model contains special timestep indicators
    model_display_name = new_names.get(model, model)
    # Keep timestep info for title but not for legend
    title_display_name = model_display_name
    if "ts42" in model.lower():
        title_display_name = f"{model_display_name} - TimeStep 42"
    elif "ts168" in model.lower():
        title_display_name = f"{model_display_name} - TimeStep 168"

    if model == 'Naive':
        # Create the histogram for the specified model's MAPE
        ax = sns.histplot(dataset[f'MAPE'] * 100, bins=np.arange(0, 101, 10), 
                        color='limegreen', alpha=0.7, stat='percent')
        
        # Add vertical line for this model's median MAPE
        model_median = dataset[f'MAPE'].median() * 100
        plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
                    label=f'{model_display_name}: {model_median:.2f}%')
        
    else:
        # Extract the relevant MAPE columns from the dataset based on rw parameter
        if rw is not None:
            # Filter columns that contain both 'MAPE_' and the rolling window value
            mape_columns = [col for col in dataset.columns if 'MAPE_' in col and f'_{rw}' in col]
        else:
            # Extract all MAPE columns from the dataset
            mape_columns = [col for col in dataset.columns if 'MAPE_' in col]
        
        # Create the histogram for the specified model's MAPE
        ax = sns.histplot(dataset[f'MAPE_{model}'] * 100, bins=np.arange(0, 101, 10), 
                        color='limegreen', alpha=0.7, stat='percent')
        
        # Plot MAPE for all models in different colors
        colors = plt.cm.tab10.colors  # Use a colormap for different models
        all_medians = []
        
        for i, col in enumerate(mape_columns):
            # Skip the current model since it's already plotted
            if col == f'MAPE_{model}':
                continue
                
            model_name = col.replace('MAPE_', '')
            # Get the display name without timestep for legend
            legend_display_name = new_names.get(model_name, model_name)
            median_val = dataset[col].median() * 100
            all_medians.append(median_val)
            
            plt.axvline(x=median_val, color=colors[i % len(colors)], linestyle='--', linewidth=1.5,
                    label=f'{legend_display_name}: {median_val:.2f}%')
        
        # Add vertical line for this model's median MAPE
        model_median = dataset[f'MAPE_{model}'].median() * 100
        plt.axvline(x=model_median, color='blue', linestyle='-', linewidth=2,
                    label=f'{model_display_name}: {model_median:.2f}%')
    
    # Set title and labels - use title_display_name with timestep info
    plt.title(f'{title_display_name} - {product} - {product_sign} - {price_type}', fontsize=20)
    plt.xlabel('MAPE (%)', fontsize=16)
    plt.ylabel('Frequency (%)', fontsize=16)
    
    # Set axis limits to ensure zero point alignment
    plt.xlim(0, 100)
    plt.ylim(0, 100)
    
    # Set ticks
    plt.xticks(np.arange(0, 101, 10), fontsize=12)
    plt.yticks(range(0, 101, 10), fontsize=12)
    
    # Add grid lines
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Save the median values and label texts for sorting
    median_labels = []
    
    # Add this model's median to the list
    median_labels.append((model_median, f'{model_display_name}: {model_median:.2f}%', 'blue', '-', 2))
    
    if model != 'Naive':
        # Add other models' medians
        for i, col in enumerate(mape_columns):
            if col == f'MAPE_{model}':
                continue
                
            model_name = col.replace('MAPE_', '')
            # Get the display name without timestep info for legend
            legend_display_name = new_names.get(model_name, model_name)
            median_val = dataset[col].median() * 100
            median_labels.append((median_val, f'{legend_display_name}: {median_val:.2f}%', 
                                colors[i % len(colors)], '--', 1.5))
    
    # Sort by median value (ascending)
    median_labels.sort(key=lambda x: x[0])

    # Create sorted legend handles and labels
    handles = []
    labels = []
    
    for median_val, label, color, style, width in median_labels:
        # Create a Line2D object for the legend
        handle = mlines.Line2D([], [], color=color, linestyle=style, linewidth=width)
        handles.append(handle)
        labels.append(label)

    # Add a note about lines representing medians
    handles.append(mlines.Line2D([], [], color='gray', linestyle='-', linewidth=0))
    labels.append("All lines represent median MAPE values")

    # Add the legend with sorted items
    plt.legend(handles=handles, labels=labels, fontsize=14, loc='best')
    
    plt.tight_layout()
    
    # Construct the filename
    filename = f'{model.lower()}_{product}_{product_sign}_{price_type}_histogram.png'
        
    plt.savefig(
        output_path / filename,
        dpi=300, bbox_inches='tight'
    )
    plt.close()

In [ ]:
# Define the path for saving plot outputs
plot_output_path = BASE_OUTPUT_PATH / 'plots'

# Create the plots directory if it doesn't exist
if not os.path.exists(plot_output_path):
    os.makedirs(plot_output_path)

# Define model groups and their result file name patterns
model_files = {
    'expanding_window_statistical': best_ex_statistical,
    'rolling_window_statistical': best_rw_statistical,
    'expanding_window_lstm_42': best_ex_lstm_42,
    'expanding_window_lstm_168': best_ex_lstm_168
}

# Add a naive model case
model_files['naive'] = best_ex_statistical

for price_type in price_types:
    # Process and plot each model group
    for file_prefix, best_model_info in model_files.items():
        product = best_model_info['PRODUCT']
        product_sign = best_model_info['PRODUCT_SIGN']
        model = best_model_info['MODEL'] if file_prefix != 'naive' else 'Naive'
        
        # For the naive model, create and save results
        if file_prefix == 'naive':
            # Get base data from ex_statistical results
            results_path = postprocessed_data_path / f'expanding_window_statistical_{product}_{product_sign}_{price_type}_results.csv'
            
            if os.path.exists(results_path):
                base_data = pd.read_csv(results_path, parse_dates=['DATE'], index_col=['DATE'])
                
                # Create naive model (forecast = previous day's actual value)
                naive_results = base_data[['ACTUAL_VALUE', 'MARGINAL_PRICE']].copy()
                naive_results['D+1'] = naive_results['ACTUAL_VALUE'].shift(1)
                naive_results = naive_results.dropna()
                
                # Calculate error metrics
                naive_results['RMSE'] = np.sqrt((naive_results['D+1'] - naive_results['ACTUAL_VALUE'])**2)
                naive_results['MAPE'] = np.abs((naive_results['D+1'] - naive_results['ACTUAL_VALUE']) / 
                                            naive_results['ACTUAL_VALUE'])
                naive_results['MAE'] = np.abs(naive_results['D+1'] - naive_results['ACTUAL_VALUE'])
                
                # Save naive results
                naive_file = postprocessed_data_path / f'naive_{product}_{product_sign}_{price_type}_results.csv'
                naive_results.to_csv(naive_file, index=True)
                
                # Plot histogram for naive model
                plot_histogram(naive_results, 'Naive', product, product_sign, price_type, plot_output_path)
        else:
            # Load model results file
            results_path = postprocessed_data_path / f'{file_prefix}_{product}_{product_sign}_{price_type}_results.csv'
            
            if os.path.exists(results_path):
                results_df = pd.read_csv(results_path, parse_dates=['DATE'], index_col=['DATE'])
            
            # For rolling window models, create histograms for each window size
            if 'rolling_window' in file_prefix:
                # Create histograms for each rolling window size (3m, 6m, 12m)
                for window_size in ['3m', '6m', '12m']:
                    # Use the base model name without the window size suffix
                    base_model_name = model.replace('_12m', '').replace('_6m', '').replace('_3m', '')
                    # Create the full model name with the current window size
                    rw_model = f"{base_model_name}_{window_size}"
                    # Plot histogram for this specific rolling window size
                    plot_histogram(results_df, rw_model, product, product_sign, price_type, plot_output_path, window_size)
            else:
                # For other models, plot a single histogram
                plot_histogram(results_df, model, product, product_sign, price_type, plot_output_path)

In [11]:
def plot_distribution(statistical_dataset, lstm_dataset, statistical_model, lstm_model, product, product_sign, price_type, output_path):
    """
    Plots the distribution of forecasts from statistical and LSTM models against actual values for the first 
    and last 3 months of the common time period in two separate graphs.
    Parameters:
    -----------
    statistical_dataset : pandas.DataFrame
        DataFrame containing the statistical model forecasts with a DatetimeIndex.
    lstm_dataset : pandas.DataFrame
        DataFrame containing the LSTM model forecasts with a DatetimeIndex.
    statistical_model : str
        Name of the statistical model used for forecasting.
    lstm_model : str
        Name of the LSTM model used for forecasting.
    product : str
        The product being forecasted (e.g., 'NCG', 'TTF').
    product_sign : str
        Sign or identifier for the product.
    price_type : str
        Type of price being analyzed (e.g., 'Bid', 'Ask').
    output_path : pathlib.Path
        Directory path where the generated plots will be saved.
    Notes:
    ------
    - Both plots have a fixed y-axis range from 0 to 30 with ticks every 5 units.
    - The plots use a 5-day interval for the x-axis.
    - The function saves two PNG files: one for the first 3 months and one for the last 3 months
      of the common date range between the two datasets.
    - The function uses a dictionary 'new_names' to map model names to display names in the plots.
    """
    # Get first 3 months and last 3 months of data (approximately 90 days)
    first_date_stat = statistical_dataset.index.min()
    last_date_stat = statistical_dataset.index.max()
    first_date_lstm = lstm_dataset.index.min()
    last_date_lstm = lstm_dataset.index.max()
    
    # Determine the common date range
    earliest_start = max(first_date_stat, first_date_lstm)
    latest_end = min(last_date_stat, last_date_lstm)
    
    # Calculate time periods
    three_months_from_start = earliest_start + pd.Timedelta(days=90)
    three_months_before_end = latest_end - pd.Timedelta(days=90)
    
    # Filter datasets for first and last 3 months
    stat_first = statistical_dataset[(statistical_dataset.index >= earliest_start) & 
                                    (statistical_dataset.index <= three_months_from_start)].copy()
    lstm_first = lstm_dataset[(lstm_dataset.index >= earliest_start) & 
                             (lstm_dataset.index <= three_months_from_start)].copy()
    
    stat_last = statistical_dataset[(statistical_dataset.index >= three_months_before_end) & 
                                   (statistical_dataset.index <= latest_end)].copy()
    lstm_last = lstm_dataset[(lstm_dataset.index >= three_months_before_end) & 
                            (lstm_dataset.index <= latest_end)].copy()
    
    # Reset indices to get DATE as column for plotting
    stat_first = stat_first.reset_index()
    lstm_first = lstm_first.reset_index()
    stat_last = stat_last.reset_index()
    lstm_last = lstm_last.reset_index()
    
    # Plot the first 3 months
    plt.figure(figsize=(24, 8))
    sns.lineplot(
        data=stat_first,
        x='DATE',
        y=f'D+1_{statistical_model}',
        label=f'{new_names.get(statistical_model, statistical_model)}',
        color='blue'
    )
    sns.lineplot(
        data=lstm_first,
        x='DATE',
        y=f'D+1_{lstm_model}',
        label=f'{new_names.get(lstm_model, lstm_model)}',
        color='orange'
    )
    sns.lineplot(
        data=lstm_first,  # We can use either dataset for actual values
        x='DATE',
        y='ACTUAL_VALUE',
        label='Actual Value',
        color='green'
    )
    plt.title(f'{new_names.get(statistical_model, statistical_model)} vs {new_names.get(lstm_model, lstm_model)} - {product} - {product_sign} - {price_type}', fontsize=16)
    plt.xlabel('Date', fontsize=14)
    plt.ylabel('Value [EUR/MW/h]', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylim(0, 30)
    plt.yticks(range(0, 31, 5))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(
        output_path / f'{statistical_model.lower()}_{lstm_model.lower()}_{product}_{product_sign}_{price_type}_first_3_months.png',
        dpi=300, bbox_inches='tight'
    )
    plt.close()
    
    # Plot the last 3 months
    plt.figure(figsize=(24, 8))
    sns.lineplot(
        data=stat_last,
        x='DATE',
        y=f'D+1_{statistical_model}',
        label=f'{new_names.get(statistical_model, statistical_model)}',
        color='blue'
    )
    sns.lineplot(
        data=lstm_last,
        x='DATE',
        y=f'D+1_{lstm_model}',
        label=f'{new_names.get(lstm_model, lstm_model)}',
        color='orange'
    )
    sns.lineplot(
        data=lstm_last,  # We can use either dataset for actual values
        x='DATE',
        y='ACTUAL_VALUE',
        label='Actual Value',
        color='green'
    )
    plt.title(f'{new_names.get(statistical_model, statistical_model)} vs {new_names.get(lstm_model, lstm_model)} - {product} - {product_sign} - {price_type}', fontsize=16)
    plt.xlabel('Date', fontsize=14)
    plt.ylabel('Value [EUR/MW/h]', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylim(0, 30)
    plt.yticks(range(0, 31, 5))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(
        output_path / f'{statistical_model.lower()}_{lstm_model.lower()}_{product}_{product_sign}_{price_type}_last_3_months.png',
        dpi=300, bbox_inches='tight'
    )
    plt.close()

In [ ]:
# Get product and product_sign from the best expanding window statistical model
product = best_ex_statistical['PRODUCT']
product_sign = best_ex_statistical['PRODUCT_SIGN']

# Loop through different price types (AVERAGE and MARGINAL)
for price_type in price_types:
    # Define paths to the CSV files containing model results
    # For statistical models
    ex_statistical_path = postprocessed_data_path / f'expanding_window_statistical_{product}_{product_sign}_{price_type}_results.csv'
    # For LSTM models with 42-hour timestep
    ex_lstm_42_path = postprocessed_data_path / f'expanding_window_lstm_42_{product}_{product_sign}_{price_type}_results.csv'

    # Call the plot_distribution function to create comparison charts
    # This function will create two plots:
    # 1. Comparing statistical vs LSTM forecasts for the first 3 months
    # 2. Comparing statistical vs LSTM forecasts for the last 3 months
    plot_distribution(
        # Load statistical model results with date as index
        pd.read_csv(ex_statistical_path, parse_dates=['DATE'], index_col=['DATE']),
        # Load LSTM model results with date as index
        pd.read_csv(ex_lstm_42_path, parse_dates=['DATE'], index_col=['DATE']),
        # Use the best statistical model based on revenue
        best_ex_statistical['MODEL'],
        # Use the best LSTM model with 42-hour timestep based on revenue
        best_ex_lstm_42['MODEL'],
        # Pass other parameters needed for file naming and chart titles
        product,
        product_sign,
        price_type,
        # Directory where plots will be saved
        plot_output_path
    )

In [14]:
def plot_models_comparison(dataset, price_type, metric, output_folder):
    """
    Creates a comparison plot of top-performing models across different product blocks and signs.
    This function visualizes the performance of the top 3 models from each model group 
    (EXPANDING_WINDOW_STATISTICAL, ROLLING_WINDOW_STATISTICAL, EXPANDING_WINDOW_LSTM) 
    for a specific price type and metric. Models are ranked within their groups and plotted 
    with different markers based on their rank.
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset containing model comparison data with columns:
        'PRICE_TYPE', 'MODEL_GROUP', 'MODEL_NAME', 'PRODUCT', 'PRODUCT_SIGN', 
        and the specified metric column.
    price_type : str
        The capacity price type to filter by (e.g., 'Unitary', 'Marginal').
    metric : str
        The performance metric to plot, such as 'REVENUE' or 'COMBINED_SCORE'.
        For 'REVENUE', higher values are better.
        For error metrics like 'COMBINED_SCORE', higher values are also better in normalized scores.
    output_folder : pathlib.Path
        The folder path where the plot image will be saved.
    Returns:
    --------
    None
        The function saves the plot as an image file and closes the plot.
    Notes:
    ------
    - For each model group, the top 3 performing models are selected based on the average
      value of the specified metric.
    - Different markers are used to indicate the rank of models: 'x' for 1st, 'o' for 2nd, 
      and 's' (square) for 3rd place.
    - Different colors represent different model groups.
    - For 'REVENUE' metric, y-axis ranges from 0 to 10,000 with 1,000 increments.
    - For 'COMBINED_SCORE' metric, y-axis is fixed between 0.99 and 1.0.
    - The plot is saved as a PNG file with naming convention: 
      'model_comparison_{price_type}_{metric}.png'
    """
    # Create figure
    plt.figure(figsize=(12, 12))

    # Filter the dataset for the specified price type
    dataset = dataset.loc[dataset['PRICE_TYPE'] == price_type]

    # Define model groups and assign different colors to each group
    model_groups = dataset['MODEL_GROUP'].unique()
    group_colors = {
        'EXPANDING_WINDOW_STATISTICAL': 'blue',
        'ROLLING_WINDOW_STATISTICAL': 'green',
        'EXPANDING_WINDOW_LSTM': 'red',
    }
    
    products = dataset['PRODUCT'].unique()
    product_signs = dataset['PRODUCT_SIGN'].unique()

    # Create x-axis labels that combine product and sign
    x_labels = [f"{product}_{sign}" for product in products for sign in product_signs]
    
    # Create a mapping of x-axis positions
    x_positions = {}
    for j, (product, sign) in enumerate([(p, s) for p in products for s in product_signs]):
        x_positions[(product, sign)] = j

    # Select top 3 models from each group based on the metric
    top_models_by_group = {}
    for group in model_groups:
        group_data = dataset[dataset['MODEL_GROUP'] == group]
        if metric == 'REVENUE':
            # For revenue, higher is better
            top_models = group_data.groupby('MODEL_NAME')[metric].mean().nlargest(3).index.tolist()
        else:
            # For error metrics like COMBINED_SCORE, higher is better in our normalized scores
            top_models = group_data.groupby('MODEL_NAME')[metric].mean().nlargest(3).index.tolist()
        
        # Store top models in order (1st, 2nd, 3rd)
        top_models_by_group[group] = top_models

    # Define markers for 1st, 2nd, and 3rd place models
    rank_markers = ['x', 'o', 's']  # 1st: x, 2nd: o, 3rd: square
    
    # Filter the dataset to only include top models
    top_models_list = [model for group_models in top_models_by_group.values() for model in group_models]
    filtered_dataset = dataset[dataset['MODEL_NAME'].isin(top_models_list)]
    
    # Track plotted models and store handles for legend sorting
    handles_by_group = {group: [] for group in model_groups}
    labels_by_group = {group: [] for group in model_groups}
    
    # First plot data points for all models
    for i, (index, row) in enumerate(filtered_dataset.iterrows()):
        model = row['MODEL_NAME']
        model_group = row['MODEL_GROUP']
        product = row['PRODUCT']
        sign = row['PRODUCT_SIGN']
        
        if metric in row and not pd.isna(row[metric]):
            x_pos = x_positions[(product, sign)]
            value = row[metric]
            
            # Get the rank of the model within its group
            group_models = top_models_by_group[model_group]
            if model in group_models:
                rank = group_models.index(model)  # 0 for 1st, 1 for 2nd, 2 for 3rd
                marker = rank_markers[rank]
            else:
                marker = '.'  # Default marker if not in top 3 (shouldn't happen)
            
            # Plot with appropriate color for the model group and marker based on rank
            scatter = plt.scatter(x_pos, value, 
                    s=100,  # Larger size of the marker for visibility
                    color=group_colors.get(model_group, 'blue'),
                    marker=marker,
                    label=model,
                    alpha=0.7)
            
            # Store handles and labels for later sorting by group
            if model not in labels_by_group[model_group]:
                handles_by_group[model_group].append(scatter)
                labels_by_group[model_group].append(model)

    # Set appropriate title based on metric
    if metric == 'REVENUE':
        title = f'Revenue by Top 3 Models per Group, Block and Sign for {price_type} capacity prices'
        ylabel = 'Revenue [EUR/MW/h]'
        # For revenue, use fixed y-ticks from 0 to 10000 by 1000
        plt.yticks(np.arange(0, 10001, 1000))
    elif metric == 'COMBINED_SCORE':
        title = f'Combined Error Score by Top 3 Models per Group, Block and Sign for {price_type} capacity prices'
        ylabel = 'Combined Score'
        # Set fixed y-range between 0.98 and 1.0 for combined score
        plt.ylim(0.99, 1.0)
        # Create appropriate ticks within this range
        plt.yticks(np.linspace(0.99, 1.0, 11))
    else:
        title = f'{metric} by Top 3 Models per Group, Block and Sign for {price_type} capacity prices'
        ylabel = metric

    plt.title(title, fontsize=16)
    plt.xlabel('Time Block and Sign', fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    plt.xticks(range(len(x_labels)), x_labels, rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Create a better legend sorted by group then by performance within group
    all_handles = []
    all_labels = []
    
    # Sort the groups for consistent order
    sorted_groups = sorted(model_groups)
    
    # Add a header for each group and then the models in that group
    for group in sorted_groups:
        # Get models for this group
        group_models = top_models_by_group[group]
        
        # Sort handles and labels based on the rank in top_models_by_group
        sorted_indices = []
        for model in group_models:
            if model in labels_by_group[group]:
                idx = labels_by_group[group].index(model)
                sorted_indices.append(idx)
        
        sorted_handles = [handles_by_group[group][i] for i in sorted_indices]
        sorted_labels = [labels_by_group[group][i] for i in sorted_indices]
        
        # Add to overall lists
        all_handles.extend(sorted_handles)
        all_labels.extend(sorted_labels)
    
    # Create the legend inside the plot
    plt.legend(all_handles, all_labels, 
               loc='upper left',  # Place legend inside the plot
               bbox_to_anchor=(0.01, 0.99),  # Position at the top-left corner
               fontsize=10,
               ncol=1)  # Single column legend
    
    plt.tight_layout()
    
    # Save the plot with appropriate filename based on metric
    metric_str = str(metric).lower()
    plt.savefig(output_folder / f'model_comparison_{price_type}_{metric_str}.png', 
               dpi=300, bbox_inches='tight')
    plt.close()

In [17]:
all_models_comparison_path = postprocessed_data_path / 'all_models_comparison.csv'
all_models_comparison = pd.read_csv(all_models_comparison_path)
plot_models_comparison(all_models_comparison, 'AVERAGE', 'REVENUE', plot_output_path)
plot_models_comparison(all_models_comparison, 'MARGINAL', 'REVENUE', plot_output_path)